In [ ]:
from pathlib import Path
from time import perf_counter
import math
import pickle

import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import torch

from foamacoustics import (
    F1ASolver,
    pressure_to_oaspl,
    pressure_to_spl_spectrum,
    spl_spectrum_to_a_weighted_oaspl,
    spl_spectrum_to_oaspl,
)
from foamacoustics.plotting import (
    plot_observer_spectra,
    plot_result_pressure_comparison,
)

plt.rcParams.update({"figure.dpi": 120, "axes.grid": True, "grid.alpha": 0.25})

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Run the notebook from the FoamAcoustics repository root. Change data_dir if
# the NPZ archives are stored elsewhere.
repo_dir = Path.cwd().resolve()
if not (repo_dir / "pyproject.toml").is_file():
    raise FileNotFoundError(
        "Run this notebook from the FoamAcoustics repository root, or update repo_dir/data_dir."
    )
data_dir = repo_dir.parent
real_npz = data_dir / "p_surface_10x7E_7000RPM_AMI.npz"
gno_npz = data_dir / "10X7E_7000_surface_time_prediction.npz"
measurement_dir = repo_dir / "actual"
figure_dir = repo_dir / "propeller_plane_spl_comparison_figures"
figure_dir.mkdir(parents=True, exist_ok=True)
for path in (real_npz, gno_npz):
    if not path.is_file():
        raise FileNotFoundError(path)
if not measurement_dir.is_dir():
    raise FileNotFoundError(measurement_dir)

rpm = 7000.0
rotation_frequency_hz = rpm / 60.0
blade_count = 2  # Change this if a different physical blade count is intended.
blade_passing_frequency_hz = blade_count * rotation_frequency_hz
rotation_center_m = [0.0, 0.0, 0.0]
omega_rad_s = [0.0, 2.0 * math.pi * rotation_frequency_hz, 0.0]

# Azimuth is measured in the X–Z propeller plane from +X toward +Z.
observer_radius_m = 1.88
observer_angles_deg = torch.arange(
    90.0, -91.0, -10.0, device=device, dtype=torch.float64
)
observer_angles_rad = torch.deg2rad(observer_angles_deg)
observers_m = observer_radius_m * torch.stack(
    (
        torch.cos(observer_angles_rad),
        torch.sin(observer_angles_rad),
        torch.zeros_like(observer_angles_rad),
    ),
    dim=1,
)

density_kg_m3 = 1.225
sound_speed_m_s = 343.0
response_delay_s = 1.0e-5
chunk_size = 64

# Both archives contain about ten rotations. The final nine rotations are
# fully arrived at this observer and give exact shaft/BPF FFT bins.
spectrum_rotations = 9
samples_per_rotation = 360
spectrum_frequency_ceiling_hz = 5000.0
measurement_record_key = 10

print(f"Device: {device}")
print(f"Observer: {observers_m.tolist()} m")
print(f"Shaft frequency: {rotation_frequency_hz:g} Hz")
print(
    f"Blade-passing frequency ({blade_count} blades): {blade_passing_frequency_hz:g} Hz"
)
print(f"Angular velocity: {omega_rad_s} rad/s")
print(f"Measurement folder: {measurement_dir}")

## 1. Configure the two pressure archives

The real legacy archive stores absolute kinematic pressure and inward-facing normals. It is converted to gauge pressure in pascals by multiplying by `rho` and subtracting `101325 × rho`, then its normals are flipped. The compact GNO archive supplies kinematic-gauge pressure metadata and outward-facing normals, so its predicted pressure can be read directly with automatic unit conversion.

In [ ]:
common_solver_options = dict(
    rpm=rpm,
    permeable=False,
    moving_surface=True,
    rotation_center_m=rotation_center_m,
    omega_rad_s=omega_rad_s,
    sound_speed_m_s=sound_speed_m_s,
    response_delay_s=response_delay_s,
    adjust_time_step=True,
    device=device,
    chunk_size=chunk_size,
)

real_solver = F1ASolver.from_npz(
    real_npz,
    pressure_field="p_surface_true",
    pressure_units="kinematic",
    density_kg_m3=density_kg_m3,
    pressure_reference_pa=101325.0 * density_kg_m3,
    flip_normals=True,
    **common_solver_options,
)
gno_solver = F1ASolver.from_npz(
    gno_npz,
    pressure_field="p_surface_pred",
    pressure_units="auto",
    density_kg_m3=density_kg_m3,
    pressure_reference_pa=0.0,
    flip_normals=False,
    **common_solver_options,
)

solvers = {
    "CFD (AMI)": real_solver,
    "GNO": gno_solver,
}

In [ ]:
for label, solver in solvers.items():
    series = solver.series
    first_frame = next(series.iter_frames())
    assert first_frame.face_centers_m is not None
    assert first_frame.face_normals is not None
    assert first_frame.surface_velocity_m_s is not None
    assert torch.isfinite(first_frame.pressure_pa).all()
    assert torch.isfinite(first_frame.face_normals).all()

    time_steps_s = torch.diff(series.absolute_times_s)
    normal_error = torch.max(
        torch.abs(torch.linalg.vector_norm(first_frame.face_normals, dim=1) - 1.0)
    )
    maximum_speed_m_s = torch.linalg.vector_norm(
        first_frame.surface_velocity_m_s, dim=1
    ).max()
    print(f"\n{label}")
    print(f"  Archive: {series.path}")
    print(f"  Schema/field: {series.schema} / {series.pressure_field}")
    print(f"  Frames/faces: {len(series):,} / {series.panel_count:,}")
    print(f"  Duration: {float(series.source_times_s[-1]):.9f} s")
    print(
        "  Time-step range: "
        f"{float(time_steps_s.min()) * 1.0e6:.6f} to "
        f"{float(time_steps_s.max()) * 1.0e6:.6f} µs"
    )
    print(
        "  First-frame pressure range: "
        f"{float(first_frame.pressure_pa.min()):.6g} to "
        f"{float(first_frame.pressure_pa.max()):.6g} Pa"
    )
    print(f"  Maximum face speed: {float(maximum_speed_m_s):.6f} m/s")
    print(f"  Maximum unit-normal error: {float(normal_error):.3e}")

## 2. Evaluate the moving-surface Farassat 1A pressure

The source mesh, normals, face velocity, Doppler factors, and retarded arrival times all vary with rotor angle. Moving-surface propagation therefore uses the nonuniform arrival interpolation path.

In [ ]:
def solve_case(label, solver):
    progress = tqdm(total=len(solver.series), desc=label, unit="frame")

    def report_progress(done, total, message):
        progress.update(done - progress.n)
        if done == total:
            progress.close()

    start = perf_counter()
    try:
        result = solver.predict(
            observers_m.to(device),
            progress_callback=report_progress,
            progress_every=25,
            remove_mean=False,
            algorithm="auto",
        )
    finally:
        progress.close()
    elapsed_s = perf_counter() - start

    observer_count = 1 if observers_m.ndim == 1 else observers_m.shape[0]
    expected_shape = (
        (len(solver.series),)
        if observer_count == 1 and observers_m.ndim == 1
        else (observer_count, len(solver.series))
    )
    assert result.total_pressure_pa.shape == expected_shape
    assert torch.isfinite(result.thickness_pressure_pa).all()
    assert torch.isfinite(result.loading_pressure_pa).all()
    assert torch.isfinite(result.total_pressure_pa).all()
    torch.testing.assert_close(
        result.total_pressure_pa,
        result.thickness_pressure_pa + result.loading_pressure_pa,
    )

    maximum_delay_s = float(torch.as_tensor(result.maximum_delay_s).max())
    print(
        f"{label}: {result.propagation_algorithm}, "
        f"maximum relative delay={maximum_delay_s * 1.0e3:.6f} ms, "
        f"wall time={elapsed_s:.2f} s"
    )
    return result


results = {label: solve_case(label, solver) for label, solver in solvers.items()}


def observer_matrix(values):
    values = torch.as_tensor(values)
    return values[None, :] if values.ndim == 1 else values


result_observers_m = (
    observer_matrix(next(iter(results.values())).observer_m).detach().cpu()
)
observer_count = result_observers_m.shape[0]
observer_labels = [
    f"Observer {index}: ({float(position[0]):.3f}, {float(position[1]):.3f}, {float(position[2]):.3f}) m"
    for index, position in enumerate(result_observers_m)
]
print(f"Solved {observer_count} observer(s).")

In [ ]:
colors = {"CFD (AMI)": "black", "GNO": "tab:blue"}
pressure_plot = plot_result_pressure_comparison(
    results,
    observer_labels=observer_labels,
    colors=colors,
    time_scale=1.0e3,
    time_unit="ms",
    shade_incomplete=True,
    figsize=(10.0, max(4.0, 2.8 * observer_count)),
)
pressure_plot.figure.savefig(
    figure_dir / "pressure_histories.png", dpi=300, bbox_inches="tight"
)
plt.show()

## 3. Compute comparable SPL spectra

The spectra use the final nine rotations, after every face contribution has reached the observer. Both histories are linearly resampled to an end-exclusive grid with 360 samples per rotation (36 kHz). The 3240-sample window contains exactly nine shaft rotations, so the 100 Hz shaft frequency and 200 Hz two-blade passing frequency lie exactly on FFT bins. No window is applied; `pressure_to_spl_spectrum` returns single-sided RMS pressure per bin.

In [ ]:
spectrum_sampling_rate_hz = samples_per_rotation * rotation_frequency_hz
spectrum_sample_count = spectrum_rotations * samples_per_rotation
common_end_s = min(float(result.observer_times_s[-1]) for result in results.values())
common_start_s = common_end_s - spectrum_rotations / rotation_frequency_hz
maximum_delay_s = max(
    float(torch.as_tensor(result.maximum_delay_s).max()) for result in results.values()
)
if common_start_s < maximum_delay_s:
    raise RuntimeError(
        "The fully arrived observer history is too short for the requested spectrum window."
    )

uniform_time_s = (
    common_start_s
    + torch.arange(spectrum_sample_count, dtype=torch.float64)
    / spectrum_sampling_rate_hz
)


def resample_pressure(result, target_time_s):
    source_time_s = result.observer_times_s.detach().cpu().contiguous()
    source_pressure_pa = observer_matrix(result.total_pressure_pa).detach().cpu()
    target_time_s = target_time_s.contiguous()
    if target_time_s[0] < source_time_s[0] or target_time_s[-1] > source_time_s[-1]:
        raise ValueError(
            "The requested uniform grid lies outside the observer history."
        )
    right = torch.searchsorted(source_time_s, target_time_s).clamp(
        1, source_time_s.numel() - 1
    )
    left = right - 1
    weight = (target_time_s - source_time_s[left]) / (
        source_time_s[right] - source_time_s[left]
    )
    return source_pressure_pa[:, left] + weight[None, :] * (
        source_pressure_pa[:, right] - source_pressure_pa[:, left]
    )


uniform_pressure_pa = {
    label: resample_pressure(result, uniform_time_s)
    for label, result in results.items()
}
spectra = {
    label: [
        pressure_to_spl_spectrum(
            observer_pressure_pa,
            sampling_rate_hz=spectrum_sampling_rate_hz,
            remove_mean=True,
        )
        for observer_pressure_pa in pressure_pa
    ]
    for label, pressure_pa in uniform_pressure_pa.items()
}


def measurement_path_for_elevation(elevation_deg):
    elevation = int(round(float(elevation_deg)))
    if elevation > 0:
        angle_token = f"p{elevation}"
    elif elevation < 0:
        angle_token = f"m{abs(elevation)}"
    else:
        angle_token = "00"
    return measurement_dir / f"10x7E_{angle_token}_actual.pkl"


def resample_measurement_window(
    pressure_pa,
    source_sampling_rate_hz,
    target_sampling_rate_hz,
    target_sample_count,
    filter_taps=257,
):
    """Return a centered, anti-aliased measurement window on the target grid."""
    pressure_pa = torch.as_tensor(pressure_pa, dtype=torch.float64)
    duration_s = target_sample_count / target_sampling_rate_hz
    center_s = pressure_pa.numel() / (2.0 * source_sampling_rate_hz)
    target_time_s = (
        center_s
        - 0.5 * duration_s
        + torch.arange(target_sample_count, dtype=torch.float64)
        / target_sampling_rate_hz
    )

    # Low-pass before downsampling. The cutoff lies below the target Nyquist
    # frequency and well above the plotted 5 kHz comparison band.
    half_width = filter_taps // 2
    cutoff_hz = 0.45 * target_sampling_rate_hz
    offsets = torch.arange(-half_width, half_width + 1, dtype=torch.float64)
    kernel = (
        2.0
        * cutoff_hz
        / source_sampling_rate_hz
        * torch.sinc(2.0 * cutoff_hz * offsets / source_sampling_rate_hz)
    )
    kernel *= torch.hann_window(filter_taps, periodic=False, dtype=torch.float64)
    kernel /= kernel.sum()

    first_index = max(
        0, int(torch.floor(target_time_s[0] * source_sampling_rate_hz)) - half_width - 2
    )
    last_index = min(
        pressure_pa.numel(),
        int(torch.ceil(target_time_s[-1] * source_sampling_rate_hz)) + half_width + 3,
    )
    source_segment = pressure_pa[first_index:last_index]
    filtered_segment = torch.nn.functional.conv1d(
        source_segment[None, None, :],
        kernel[None, None, :],
        padding=half_width,
    )[0, 0]
    source_time_s = (
        torch.arange(first_index, last_index, dtype=torch.float64)
        / source_sampling_rate_hz
    )
    right = torch.searchsorted(source_time_s, target_time_s).clamp(
        1, source_time_s.numel() - 1
    )
    left = right - 1
    weight = (target_time_s - source_time_s[left]) / (
        source_time_s[right] - source_time_s[left]
    )
    return filtered_segment[left] + weight * (
        filtered_segment[right] - filtered_segment[left]
    )


measurement_spectra = []
measurement_pressure_windows_pa = []
measurement_paths = []
for elevation_deg in observer_angles_deg.detach().cpu():
    measurement_path = measurement_path_for_elevation(elevation_deg)
    if not measurement_path.is_file():
        raise FileNotFoundError(measurement_path)
    with measurement_path.open("rb") as stream:
        measurement_archive = pickle.load(stream)
    if measurement_record_key not in measurement_archive:
        raise KeyError(
            f"Record {measurement_record_key!r} is absent from {measurement_path}."
        )
    measurement = measurement_archive[measurement_record_key]
    if measurement.get("unit") != "Pa":
        raise ValueError(
            f"Expected pressure in Pa in {measurement_path}, got {measurement.get('unit')!r}."
        )
    measurement_pressure_pa = torch.as_tensor(
        measurement["scaled_samples"], dtype=torch.float64
    )
    measurement_pressure_window_pa = resample_measurement_window(
        measurement_pressure_pa,
        source_sampling_rate_hz=float(measurement["sample_rate"]),
        target_sampling_rate_hz=spectrum_sampling_rate_hz,
        target_sample_count=spectrum_sample_count,
    )
    measurement_pressure_windows_pa.append(measurement_pressure_window_pa)
    measurement_spectra.append(
        pressure_to_spl_spectrum(
            measurement_pressure_window_pa,
            sampling_rate_hz=spectrum_sampling_rate_hz,
            remove_mean=True,
        )
    )
    measurement_paths.append(measurement_path)

print(
    f"Spectrum window: {common_start_s:.9f} to {common_end_s:.9f} s "
    f"({spectrum_rotations} rotations, end-exclusive)"
)
print(
    f"Samples/rate/resolution: {spectrum_sample_count:,} / "
    f"{spectrum_sampling_rate_hz:g} Hz / "
    f"{next(iter(spectra.values()))[0].frequency_resolution_hz:.9f} Hz"
)
print(
    f"Loaded {len(measurement_spectra)} measured spectra, each resampled to "
    f"{spectrum_sample_count:,} samples at {spectrum_sampling_rate_hz:g} Hz; "
    f"measurement resolution: {measurement_spectra[0].frequency_resolution_hz:.9f} Hz"
)
print("\nLevel summary")
print(
    f"{'Observer':<48} {'Case':<22} {'Time OASPL':>12} {'≤5 kHz':>12} {'≤5 kHz A-wtd':>15}"
)
for observer_index, observer_label in enumerate(observer_labels):
    for case_label, case_spectra in spectra.items():
        spectrum = case_spectra[observer_index]
        time_oaspl_db = pressure_to_oaspl(
            uniform_pressure_pa[case_label][observer_index]
        )
        band_oaspl_db = spl_spectrum_to_oaspl(
            spectrum.spl_db[1:],
            spectrum.frequencies_hz[1:],
            frequency_ceiling_hz=spectrum_frequency_ceiling_hz,
        )
        a_weighted_db = spl_spectrum_to_a_weighted_oaspl(
            spectrum.spl_db[1:],
            spectrum.frequencies_hz[1:],
            frequency_ceiling_hz=spectrum_frequency_ceiling_hz,
        )
        print(
            f"{observer_label:<48} {case_label:<22} "
            f"{float(time_oaspl_db):>11.3f} dB "
            f"{float(band_oaspl_db):>11.3f} dB "
            f"{float(a_weighted_db):>14.3f} dBA"
        )

In [ ]:
comparison_spectra = {
    "CFD (AMI)": spectra["CFD (AMI)"],
    "GNO": spectra["GNO"],
    "Measurement": measurement_spectra,
}
frequency_resolution_hz = next(iter(spectra.values()))[0].frequency_resolution_hz
spectrum_plots = plot_observer_spectra(
    comparison_spectra,
    observer_titles=[
        f"APC 10x7E @ 7000 RPM {observer_label}" for observer_label in observer_labels
    ],
    xlim_hz=(max(frequency_resolution_hz, 100.0), spectrum_frequency_ceiling_hz),
    blade_passing_frequency_hz=blade_passing_frequency_hz,
    colors={**colors, "Measurement": "tab:orange"},
    linewidths={"CFD (AMI)": 1.1, "GNO": 1.1, "Measurement": 0.9},
    alphas={"CFD (AMI)": 1.0, "GNO": 1.0, "Measurement": 0.85},
    legend_order=("CFD (AMI)", "GNO", "Measurement"),
    figsize=(10.0, 4.2),
)
for observer_index, figure in enumerate(spectrum_plots.figures):
    figure.savefig(
        figure_dir / f"spectrum_observer_{observer_index:02d}.png",
        dpi=300,
        bbox_inches="tight",
    )
plt.show()

In [ ]:
for observer_index, observer_label in enumerate(observer_labels):
    print(f"\n{observer_label}")
    print("Shaft-harmonic comparison (GNO − real)")
    print(f"{'Order':>5} {'Frequency':>10} {'Real':>10} {'GNO':>10} {'Difference':>12}")
    real_spectrum = spectra["CFD (AMI)"][observer_index]
    gno_spectrum = spectra["GNO"][observer_index]
    for order in range(1, 11):
        target_frequency_hz = order * rotation_frequency_hz
        bin_index = int(
            torch.argmin(torch.abs(real_spectrum.frequencies_hz - target_frequency_hz))
        )
        frequency_hz = float(real_spectrum.frequencies_hz[bin_index])
        real_db = float(real_spectrum.spl_db[bin_index])
        gno_db = float(gno_spectrum.spl_db[bin_index])
        bpf_marker = "*" if order % blade_count == 0 else " "
        print(
            f"{order:>4}{bpf_marker} {frequency_hz:>9.1f} "
            f"{real_db:>9.3f} {gno_db:>9.3f} "
            f"{gno_db - real_db:>+11.3f} dB"
        )
print("\n* BPF harmonic for the configured blade count")